## STEP 1 GraphRAG

In [1]:
import json
import numpy as np
import chromadb
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")

print("초기화 완료!")

초기화 완료!


### Local Search 구현

In [15]:
def extract_entities(question):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """질문에서 검색에 사용할 핵심 키워드를 추출하세요.
  인물, 조직, 국가, 사건, 주제 등을 쉼표로 구분해서 출력만 하세요.
  키워드가 없으면 질문의 핵심 명사를 추출하세요.
  절대 설명하지 말고 키워드만 출력하세요."""},
            {"role": "user", "content": question}
        ]
    )
    return [e.strip() for e in response.choices[0].message.content.split(",") if e.strip()]

def graph_local_search(question, hops=2):
    """엔티티 중심 그래프 탐색"""
    entities = extract_entities(question)
    print(f"추출된 엔티티: {entities}")

    graph_context = []

    with driver.session() as session:
        for entity in entities:
            # 1홉: 직접 연결
            result = session.run("""
                  MATCH (n {name: $name})-[r]-(m)
                  RETURN n.name AS source, type(r) AS relation, m.name AS target
              """, name=entity)

            for record in result:
                triple = f"{record['source']} -{record['relation']}-> {record['target']}"
                graph_context.append(triple)

            # 2홉: 간접 연결
            if hops >= 2:
                result = session.run("""
                      MATCH (n {name: $name})-[r1]-(mid)-[r2]-(end)
                      WHERE end.name <> n.name
                      RETURN n.name AS source, type(r1) AS r1, mid.name AS mid, type(r2) AS r2, end.name AS target
                      LIMIT 10
                  """, name=entity)

                for record in result:
                    triple = f"{record['source']} -{record['r1']}-> {record['mid']} -{record['r2']}->{record['target']}"
    graph_context.append(triple)

    # 중복 제거
    graph_context = list(set(graph_context))
    return graph_context

# 테스트
context = graph_local_search("이란 전쟁이 경제에 미치는 영향은?")
print(f"\n그래프 컨텍스트 {len(context)}개:")
for c in context:
    print(f"  {c}")


추출된 엔티티: ['이란', '전쟁', '경제', '영향']

그래프 컨텍스트 11개:
  이란 -사태로-> 국제유가
  이란 -사망한-> 아야톨라 알리 하메네이
  이란 -전쟁_여파로-> 걸프 지역 원유 공급망
  이란 -검토_중인-> 이스라엘
  이란 -폭격-> 샤자라 타이이바 초등학교
  이란 -전쟁-> 미국 -공동_연구->우리나라
  이란 -공격하다-> 이스라엘
  이란 -공습-> 미국
  이란 -저항하다-> 미국과 이스라엘
  이란 -전쟁-> 미국
  이란 -보유한-> 60% 농축 우라늄


### 벡터 검색 + 그래프 검색 결합

In [16]:
def graph_local_search(question, hops=2):
    entities = extract_entities(question)
    print(f"추출된 엔티티: {entities}")
    graph_context = []

    with driver.session() as session:
        for entity in entities:
            # 1홉 - CONTAINS로 부분 매칭
            result = session.run("""
                  MATCH (n)-[r]-(m)
                  WHERE n.name CONTAINS $name OR n.name = $name
                  RETURN n.name AS source, type(r) AS relation, m.name AS target
                  LIMIT 15
              """, name=entity)
            for record in result:
                triple = f"{record['source']} -{record['relation']}-> {record['target']}"
                graph_context.append(triple)

            # 2홉
            if hops >= 2:
                result = session.run("""
                      MATCH (n)-[r1]-(mid)-[r2]-(end)
                      WHERE (n.name CONTAINS $name OR n.name = $name) AND end.name <> n.name
                      RETURN n.name AS source, type(r1) AS r1, mid.name AS mid, type(r2) AS r2, end.name AS target
                      LIMIT 10
                  """, name=entity)
                for record in result:
                    triple = f"{record['source']} -{record['r1']}-> {record['mid']} -{record['r2']}->{record['target']}"
                    graph_context.append(triple)

    return list(set(graph_context))

answer = graph_local_search("이란 전쟁이 경제에 미치는 영향은?")
print(f"\n=== 답변 ===\n{answer}")

추출된 엔티티: ['이란', '전쟁', '경제', '영향']

=== 답변 ===
['이란 혁명수비대 -복종을_선언하다-> 모즈타바 하메네이', '하준경 경제성장수석 -참석한-> 비상경제점검회의 -참석한->김용범 정책실장', '이란 -검토_중인-> 이스라엘 -검토_중인->미국', '하준경 경제성장수석 -참석한-> 비상경제점검회의 -관련한->중동 상황', '이란 전문가회의 -발표하다-> 모즈타바 하메네이', '이란 -전쟁-> 미국 -검토_중인->이스라엘', '비상경제점검회의 -관련한-> 중동 상황', '이란 -사태로-> 국제유가', '재정경제부 -참석-> 제1차 협의체 회의', '재정경제부 -참석-> 제1차 협의체 회의 -주재->오현주', '이란 -전쟁_여파로-> 걸프 지역 원유 공급망', '이란 -폭격-> 샤자라 타이이바 초등학교', '이란 -검토_중인-> 이스라엘 -파괴한->이스파한 핵시설', '전쟁 일시 휴전 및 인질·포로 교환 -합의하다-> 이스라엘과 팔레스타인 무장정파 하마스', '비상경제점검회의 -참석한-> 청와대 강훈식 비서실장', '이란 -전쟁-> 미국 -비판하다->트럼프', '이란 -전쟁-> 미국 -비난하다->모즈타바 하메네이', '매일경제 -주최하다-> 이스트웨스트 바이오파마 서밋 서울', '이란이 한 것이다 -주장하다-> 트럼프 대통령', '이란이 민간인 목표로 삼는다 -주장하다-> 피트 헤그세스', '이란 -사망한-> 아야톨라 알리 하메네이', '이란 -전쟁-> 미국 -파괴한->이스파한 핵시설', '이란 -검토_중인-> 이스라엘', '이란 -전쟁-> 미국 -공동_연구->우리나라', '이란 -검토_중인-> 이스라엘 -공습->미국', '비상경제점검회의 -주재한-> 이재명 대통령', '이란 -공격하다-> 이스라엘', '이란 -검토_중인-> 이스라엘 -위치하다->예루살렘', '하준경 경제성장수석 -참석한-> 비상경제점검회의 -참석한->청와대 강훈식 비서실장', '이란 -공습-> 미국', '비상경제점검회의 -참석한-> 하준경 경제성장수석', '이란 

### Naive RAG vs GraphRAG 비교

In [17]:
def naive_rag(question):
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
    context = "\n\n".join([f"[참고자료 {i+1}]\n{doc}" for i, doc in enumerate(results['documents'][0])])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수  없습니다'라고 하세요."},
             {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content

questions = [
    "이란 전쟁이 한국 경제에 미치는 영향은?",
    "트럼프와 이란의 관계를 설명해줘",
    "중동 사태와 관련된 주요 인물들은?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"\n[Naive RAG]")
    print(naive_rag(q)[:200])
    print(f"\n[GraphRAG]")
    print(graph_local_search(q)[:200])
    print("=" * 60)



Q: 이란 전쟁이 한국 경제에 미치는 영향은?

[Naive RAG]
해당 정보를 찾을 수 없습니다.

[GraphRAG]
추출된 엔티티: ['이란 전쟁', '한국 경제']
[]

Q: 트럼프와 이란의 관계를 설명해줘

[Naive RAG]
트럼프와 이란의 관계는 긴장감이 지배하는 상황입니다. 트럼프 대통령은 이란의 핵 및 미사일 프로그램을 우려하며, 이란이 "미친 사람들이" 핵무기를 갖는 것에 대해 강한 반감을 보였습니다. 그는 이란 지도자들이 결국은 "죽음"을 맞이할 것이라는 경고를 하며, 이란에 대한 공습 작전의 정당성을 주장했습니다.

트럼프 행정부는 이란과의 접촉을 부인하며, 이란 측

[GraphRAG]
추출된 엔티티: ['트럼프', '이란', '관계']
['트럼프 -비판하다-> 미국 -확장->병원 공급망', '이란 혁명수비대 -복종을_선언하다-> 모즈타바 하메네이', '트럼프 -비판하다-> 미국 -공습->이란', '이란 -검토_중인-> 이스라엘 -검토_중인->미국', '트럼프 -비판하다-> 미국 -파괴한->이스파한 핵시설', '이란 전문가회의 -발표하다-> 모즈타바 하메네이', '트럼프 -비판하다-> 미국 -전쟁->이란', '이란 -전쟁-> 미국 -검토_중인->이스라엘', '도널드 트럼프 -설득하다-> 베냐민 네타냐후', '트럼프 -비판하다-> 미국 -공습->이스라엘', '트럼프 -비판하다-> 미국 -경쟁->AI', '이란 -사태로-> 국제유가', '이란 -전쟁_여파로-> 걸프 지역 원유 공급망', '이란 -폭격-> 샤자라 타이이바 초등학교', '이란 -검토_중인-> 이스라엘 -파괴한->이스파한 핵시설', '도널드 트럼프 -상담하다-> 베냐민 네타냐후', '경기도 관계자 -제보-> 경기도 누리집', '도널드 트럼프 -주도하다-> 군사 행동', '이란 -전쟁-> 미국 -비판하다->트럼프', '이란 -전쟁-> 미국 -비난하다->모즈타바 하메네이', '이란이 한 것이다 -주장하다-> 트럼프 대통령', '이란이 민간인 목표로 삼는다 -주장하다-> 피트 

## STEP 2 Global Search (커뮤니티 기반 검색)

In [18]:
import json
import chromadb
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")

### 커뮤니티 탐지

In [19]:
def detect_communities():
    """연결된 노드 그룹을 커뮤니티로 추출"""
    with driver.session() as session:
        # 모든 노드와 연결 가져오기
        result = session.run("""
              MATCH (n)-[r]-(m)
              RETURN n.name AS source, type(r) AS relation, m.name AS target
          """)

        edges = []
        for record in result:
            edges.append((record['source'], record['relation'], record['target']))

    # 간단한 Union-Find로 커뮤니티 탐지
    parent = {}

    def find(x):
        if x not in parent:
            parent[x] = x
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py

    for src, rel, tgt in edges:
        union(src, tgt)

    # 커뮤니티별 노드와 관계 정리
    communities = {}
    for src, rel, tgt in edges:
        root = find(src)
        if root not in communities:
            communities[root] = {"nodes": set(), "relations": []}
        communities[root]["nodes"].add(src)
        communities[root]["nodes"].add(tgt)
        communities[root]["relations"].append(f"{src} -{rel}-> {tgt}")

    # 크기순 정렬
    sorted_communities = sorted(communities.values(), key=lambda x: len(x["nodes"]), reverse=True)
    return sorted_communities

communities = detect_communities()
print(f"총 커뮤니티 수: {len(communities)}\n")
for i, comm in enumerate(communities[:5]):
    print(f"--- 커뮤니티 {i+1} ({len(comm['nodes'])}개 노드) ---")
    print(f"노드: {list(comm['nodes'])[:8]}")
    print(f"관계 수: {len(comm['relations'])}")
    print()

총 커뮤니티 수: 192

--- 커뮤니티 1 (36개 노드) ---
노드: ['한국', '미국과 이스라엘', '이란 전문가회의', '걸프 지역 원유 공급망', '대니얼 샤피로', '이란 최고지도자', '인핸스', '트럼프']
관계 수: 84

--- 커뮤니티 2 (20개 노드) ---
노드: ['왕과 사는 남자', '쇼박스', '서울', '국정 정상화', '오세훈', '윤석열', '서울시정', '장항준']
관계 수: 38

--- 커뮤니티 3 (19개 노드) ---
노드: ['김영배, 박주민, 전현희, 김형남', '하준경 경제성장수석', '공천 기강', '민주당', '정원오', 'AI를 통한 행정속도 강화', '이정현', '개인적인 생각']
관계 수: 38

--- 커뮤니티 4 (12개 노드) ---
노드: ['1.3%', '0.9%', '체코', '구마모토현', 'PPI', '2026 월드베이스볼클래식(WBC)', '대만', '2%']
관계 수: 24

--- 커뮤니티 5 (10개 노드) ---
노드: ['간이 약물 검사', '2003년 강남', '경찰', '김 씨', '남성', '이재룡', '음주운전 사고', '서울 강남구 청담역']
관계 수: 20



### 각 커뮤니티 요약 생성

In [20]:
def summarize_communities(communities, max_communities=10):
    """각 커뮨니티를 LLM으로 요약"""
    summaries = []

    for i, comm in enumerate(communities[:max_communities]):
        if len(comm['nodes']) < 3:  # 너무 작은 커뮤니티 스킵
            continue

        relations_text = "\n".join(comm['relations'][:20])  # 최대 20개 관계

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "아래 엔티티 관계들을 분석해서 이 그룹이 어떤 주제/사건에 대한 것인지 2-3문장으로 요약하세요."},
                     {"role": "user", "content": f"엔티티: {list(comm['nodes'])[:15]}\n\n관계:\n{relations_text}"}
            ]
        )

        summary = response.choices[0].message.content
        summaries.append({
            "id": i,
            "nodes": list(comm['nodes']),
            "summary": summary,
            "size": len(comm['nodes'])
        })
        print(f"커뮤니티 {i+1} ({len(comm['nodes'])}개 노드): {summary[:80]}...")

    return summaries

summaries = summarize_communities(communities)

커뮤니티 1 (36개 노드): 이 그룹은 이란, 미국, 이스라엘 간의 군사적 긴장과 관련된 사건을 다루고 있으며, 특히 이란의 공격과 미국 및 이스라엘 간의 공습에 대한 문제...
커뮤니티 2 (20개 노드): 이 그룹은 최근 서울의 정치적 상황 및 국민의힘 내의 갈등을 다루고 있습니다. 서울시장이자 국민의힘 소속인 오세훈과 윤석열 전 대통령 간의 정치...
커뮤니티 3 (19개 노드): 이 그룹은 서울시장 선거와 관련된 정치적 논의 및 경쟁을 다루고 있습니다. 박주민과 정원오를 중심으로 민주당의 후보 경선, 이재명 대통령과의 협...
커뮤니티 4 (12개 노드): 이 그룹은 2026 월드베이스볼클래식(WBC)와 관련된 대만 및 체코 간의 관람 관계와, 대만과 중국 간의 군사적 긴장 및 미사일 방어 시스템에...
커뮤니티 5 (10개 노드): 이 그룹은 이재룡이라는 남성이 2019년에 서울 강남구 청담역에서 발생한 음주운전 사고로 두 명의 남성을 숨지게 한 혐의를 받고 있는 사건과 관...
커뮤니티 6 (9개 노드): 이 그룹은 자율주행차 기술의 발전과 관련된 사건 및 협력 과정을 다루고 있습니다. 국토교통부가 현대차와 삼성화재와 협력하여 자율주행차 시장 확대...
커뮤니티 7 (9개 노드): 이 그룹은 부산을 중심으로 한 정치적 사건과 경제적 성과에 관한 내용이다. 박형준이 부산시장 보궐선거에서 승리하고, 부산 경제의 발전에 기여하는...
커뮤니티 8 (8개 노드): 이 엔티티 관계는 2026년 제1차 실무 마약류대책협의회와 관련된 정부 기관 및 주요 인사의 활동을 다룹니다. 이 회의는 경찰청, 대검찰청, 관...
커뮤니티 9 (7개 노드): 이 그룹은 조국과 관련된 정치적 사건 및 그의 서울시장 예비경선 출마와 관련된 내용을 다루고 있습니다. 더불어민주당에서 주최하는 서울시장 예비경...
커뮤니티 10 (7개 노드): 이 그룹은 봄동, 특히 봄동 비빔밥에 대한 관심과 그 조리법, 그리고 소비자의 선호도에 관한 주제를 다루고 있습니다. 봄동은 배추의 일종으로

### Global Search 구현

In [21]:
def graph_global_search(question):
    """커뮤니티 요약 기반 검색"""
    print(f"질문: {question}\n")

    # 모든 커뮤니티 요약을 하나로
    all_summaries = "\n\n".join([
        f"[주제 그룹 {s['id']+1}] ({s['size']}개 엔티티)\n{s['summary']}"
        for s in summaries
    ])

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """아래는 뉴스 기사들에서 추출한 주제 그룹별 요약입니다.
  이 요약들을 종합해서 질문에 포괄적으로 답변하세요.
  관련 없는 그룹은 무시하세요. 없는 내용은 지어내지 마세요."""},
            {"role": "user", "content": f"{all_summaries}\n\n질문: {question}"}
        ]
    )

    return response.choices[0].message.content

### Local Search vs Global Search vs Naive RAG 비교

In [22]:
def naive_rag(question):
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
    context = "\n\n".join(results['documents'][0])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 하세요."},
             {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content

questions = [
    "오늘 뉴스의 전체적인 흐름을 요약해줘",
    "현재 가장 중요한 이슈 3가지는?",
    "트럼프가 이란에 대해 뭐라 했어?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"\n[Naive RAG]")
    print(naive_rag(q)[:150])
    print(f"\n[Local Search]")
    print(graph_local_search(q)[:150])
    print(f"\n[Global Search]")
    print(graph_global_search(q)[:150])
    print("=" * 60)



Q: 오늘 뉴스의 전체적인 흐름을 요약해줘

[Naive RAG]
오늘 뉴스의 전체적인 흐름은 주로 비리, 부당대우, 사건사고, 그리고 미담 사례에 대한 제보를 요청하는 내용으로 시작됩니다. CBS노컷뉴스와 MBC 뉴스가 각각 제보를 받을 수 있는 연락처와 방법을 설명하며, 국민들의 참여를 통해 문제를 해결하고 소통을 강화하겠다는 의

[Local Search]
추출된 엔티티: ['뉴스', '흐름', '요약']
['CBS노컷뉴스 -제보-> 여러분']

[Global Search]
질문: 오늘 뉴스의 전체적인 흐름을 요약해줘

오늘 뉴스의 주요 흐름은 국제적인 군사적 긴장, 국내 정치적 갈등 및 서울시장 선거와 관련된 논의로 나뉘어 있습니다.

첫 번째로, 이란과 미국, 이스라엘 간의 군사적 긴장이 부각되고 있으며, 이란의 공격과 이에 대한 미국 및 이스라엘의 대응 공습이 주요 이슈로 떠오르

Q: 현재 가장 중요한 이슈 3가지는?

[Naive RAG]
현재 가장 중요한 이슈는 다음과 같습니다:

1. AI 및 에이전트 구축 시 보안 및 거버넌스 체계 구축: AI 시대에 필수적인 보안 확보와 거버넌스 체계의 필요성이 강조되고 있습니다.

2. 공급망 보안 위협: 민간 중심의 '뉴스페이스' 시대에서 위성 제작과 운영 과

[Local Search]
추출된 엔티티: ['이슈']
[]

[Global Search]
질문: 현재 가장 중요한 이슈 3가지는?

현재 가장 중요한 이슈는 다음과 같습니다:

1. **이란과 미국, 이스라엘 간의 군사적 긴장**: 이란의 군사적 행동과 미국 및 이스라엘의 공습이 격화되고 있으며, 이는 국제 유가에 미치는 영향과도 연결되어 있습니다. 이란의 군사 작전은 다른 국가들의 전략과도 연관되

Q: 트럼프가 이란에 대해 뭐라 했어?

[Naive RAG]
도널드 트럼프 미국 대통령은 이란에 대해 "미친 사람들이 핵무기를 가지면 나쁜 일이 벌어진다"며 이란의 핵·미사일 프로그램 폐기를 명분으로 시작한 공습 상황에 대해 "지금은 아주 좋